# W8 · Tuesday — Neural Networks + Data Cleaning
**PG Diploma · AI-ML & Agentic AI Engineering · IIT Gandhinagar**

---
| | |
|---|---|
| **Dataset** | `hospital_records.csv` — 2,000 patient records with 30-day readmission labels |
| **Goal** | Clean messy hospital data → Build 3-layer NumPy neural network → Cost-sensitive inference |
| **Sub-steps** | 1–2 (Data Cleaning) · 3–5 (NN Build + Evaluate) · 6–7 (Hard: accuracy trap + embeddings) |


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, average_precision_score,
    ConfusionMatrixDisplay, precision_recall_curve
)

SEED = 42
rng  = np.random.default_rng(SEED)
np.random.seed(SEED)

DATA_PATH = Path('hospital_records.csv')
print(f'NumPy  {np.__version__}  |  Pandas  {pd.__version__}')
print(f'Data file exists: {DATA_PATH.exists()}')

---
## 🟢 Sub-step 1 — Data Quality Audit

In [ ]:
# ── Load raw data ────────────────────────────────────────────────────────────
def load_raw_data(path: Path) -> pd.DataFrame:
    """Load hospital CSV; every column kept as object for inspection."""
    try:
        df = pd.read_csv(path, dtype=str, low_memory=False)
        print(f'Loaded  →  {df.shape[0]} rows × {df.shape[1]} columns')
        return df
    except FileNotFoundError:
        raise FileNotFoundError(f'Dataset not found at {path}. '
                                'Run generate_dataset.py first.')

raw = load_raw_data(DATA_PATH)
raw.head(3)

In [ ]:
# ── Audit 1: Missing values ───────────────────────────────────────────────────
def audit_missing(df: pd.DataFrame) -> pd.DataFrame:
    """Return missing-value summary per column."""
    null_count  = df.isnull().sum()
    null_pct    = (null_count / len(df) * 100).round(2)
    report      = pd.DataFrame({'missing_count': null_count,
                                'missing_pct': null_pct})
    return report[report.missing_count > 0].sort_values('missing_pct', ascending=False)

print('=== Missing Values ===')
print(audit_missing(raw).to_string())

In [ ]:
# ── Audit 2: Duplicate rows ───────────────────────────────────────────────────
def audit_duplicates(df: pd.DataFrame) -> None:
    """Report full duplicate rows (patient_id is the natural key)."""
    dup_count = df.duplicated().sum()
    pid_dups  = df.duplicated(subset='patient_id').sum()
    print(f'Full duplicate rows      : {dup_count}')
    print(f'Duplicate patient_id rows: {pid_dups}')

audit_duplicates(raw)

In [ ]:
# ── Audit 3: Age column ──────────────────────────────────────────────────────
def audit_age_column(df: pd.DataFrame) -> None:
    """Identify non-numeric, negative, and biologically impossible ages."""
    age_raw = df['age'].dropna()

    # Detect non-numeric entries
    def try_parse(val):
        try:
            return float(str(val).replace('yrs', '').replace('yr', '').strip())
        except ValueError:
            return np.nan

    age_numeric = age_raw.map(try_parse)
    unparseable = age_raw[age_numeric.isna()]

    print(f'Non-parseable age entries : {len(unparseable)}')
    print(f'Negative ages (<0)        : {(age_numeric < 0).sum()}')
    print(f'Impossible ages (>110)    : {(age_numeric > 110).sum()}')
    print(f'Valid age range found     : [{age_numeric.min():.0f}, {age_numeric.max():.0f}]')

audit_age_column(raw)

In [ ]:
# ── Audit 4: BMI column ──────────────────────────────────────────────────────
def audit_bmi_column(df: pd.DataFrame) -> None:
    """Identify zero, negative, and physiologically implausible BMI values."""
    bmi_num = pd.to_numeric(df['bmi'], errors='coerce')
    print(f'BMI == 0         : {(bmi_num == 0).sum()}')
    print(f'BMI < 10 (bad)   : {(bmi_num < 10).sum()}')
    print(f'BMI > 70 (extreme): {(bmi_num > 70).sum()}')
    print(f'BMI NaN          : {bmi_num.isna().sum()}')
    print(f'Valid BMI range  : [{bmi_num.quantile(0.01):.1f}, {bmi_num.quantile(0.99):.1f}]')

audit_bmi_column(raw)

In [ ]:
# ── Audit 5: Gender column ────────────────────────────────────────────────────
def audit_gender_column(df: pd.DataFrame) -> None:
    """Show all unique gender values (expects mixed formatting)."""
    print('Unique gender values:')
    print(df['gender'].value_counts(dropna=False).to_string())

audit_gender_column(raw)

In [ ]:
# ── Audit 6: Blood pressure (compound column) ────────────────────────────────
def audit_bp_column(df: pd.DataFrame) -> None:
    """Check format and range of blood_pressure (expected: 'sys/dia')."""
    sample = df['blood_pressure'].dropna().head(5).tolist()
    print('Sample values:', sample)

    def parse_bp(val):
        try:
            parts = str(val).split('/')
            if len(parts) == 2:
                return float(parts[0]), float(parts[1])
        except Exception:
            pass
        return np.nan, np.nan

    parsed = df['blood_pressure'].apply(parse_bp)
    sys_vals = pd.Series([p[0] for p in parsed])
    dia_vals = pd.Series([p[1] for p in parsed])
    bad_sys  = ((sys_vals < 50) | (sys_vals > 250)).sum()
    bad_dia  = ((dia_vals < 30) | (dia_vals > 150)).sum()
    print(f'Malformed / unparseable: {sys_vals.isna().sum()}')
    print(f'Out-of-range systolic  : {bad_sys}')
    print(f'Out-of-range diastolic : {bad_dia}')

audit_bp_column(raw)

In [ ]:
# ── Audit 7: Glucose column ──────────────────────────────────────────────────
def audit_glucose_column(df: pd.DataFrame) -> None:
    """Identify string labels like 'high'/'low' in glucose."""
    glucose_num = pd.to_numeric(df['glucose'], errors='coerce')
    non_numeric = df['glucose'].dropna()[glucose_num.isna()]
    print('Non-numeric glucose entries:')
    print(non_numeric.value_counts().to_string())

audit_glucose_column(raw)

In [ ]:
# ── Audit 8: Creatinine (negative values) ────────────────────────────────────
def audit_creatinine_column(df: pd.DataFrame) -> None:
    """Flag negative creatinine — physically impossible."""
    creat_num = pd.to_numeric(df['creatinine'], errors='coerce')
    print(f'Negative creatinine: {(creat_num < 0).sum()}')
    print(f'Creatinine NaN     : {creat_num.isna().sum()}')

audit_creatinine_column(raw)

In [ ]:
# ── Audit 9: Target variable distribution ────────────────────────────────────
def audit_target(df: pd.DataFrame) -> None:
    """Report class balance of readmission_30d."""
    target = pd.to_numeric(df['readmission_30d'], errors='coerce')
    vc = target.value_counts(normalize=True)
    print('Readmission_30d distribution:')
    print(vc.rename({0: 'Not Readmitted (0)', 1: 'Readmitted (1)'}).to_string())
    print(f'\nClass imbalance ratio  {vc[0]/vc[1]:.1f} : 1  → imbalanced dataset')

audit_target(raw)

### 📋 Audit Summary

| Column | Issue | Planned Fix |
|---|---|---|
| `age` | String noise (`"45 yrs"`), negatives, values > 110, ~3% missing | Strip units, numeric cast, set impossible → NaN, impute with median |
| `bmi` | Zeros (=missing), extreme outliers (>70), ~4% missing | Zero → NaN, winsorize at 1–99 pct, impute median |
| `blood_pressure` | Compound column ("120/80"), ~3% missing | Split into `systolic`/`diastolic`, impute medians |
| `gender` | Inconsistent labels: M/Male/male/1/F/Female/female/0 | Map all → 0/1 binary |
| `glucose` | String labels ("high"/"low"), ~5% missing | Map labels to numeric midpoint, impute median |
| `creatinine` | Negative values (sign flip error), ~3% missing | Abs value, impute median |
| `hba1c` | ~6% missing (highest rate) | Impute median |
| `diabetes`, `hypertension` | ~2% missing | Impute mode (binary) |
| `smoker`, `discharge_disposition` | ~2–3% missing | Impute mode |
| Duplicates | ~2% full-row duplicates | Drop duplicates keeping first |
| **Target** | Class imbalance (~6% positive) | Use PR-AUC; class weights in training |

---
## 🟢 Sub-step 2 — Principled Data Cleaning

In [ ]:
# ── Cleaning pipeline ────────────────────────────────────────────────────────

VALID_AGE_MIN  = 0
VALID_AGE_MAX  = 110
VALID_BMI_LOW  = 10.0
VALID_BMI_HIGH = 70.0
CREATININE_MAX = 15.0
GLUCOSE_LABEL_MAP = {'high': 200.0, 'low': 55.0, 'normal': 100.0}  # midpoint imputation


def clean_age(series: pd.Series) -> pd.Series:
    """Strip string noise, cast to float, set impossible values to NaN."""
    cleaned = (
        series
        .astype(str)
        .str.replace(r'\s*yrs?\s*', '', regex=True)
        .str.strip()
    )
    cleaned = pd.to_numeric(cleaned, errors='coerce')
    cleaned[(cleaned < VALID_AGE_MIN) | (cleaned > VALID_AGE_MAX)] = np.nan
    median_age = cleaned.median()
    return cleaned.fillna(median_age)


def clean_bmi(series: pd.Series) -> pd.Series:
    """Zero → NaN, winsorize extreme outliers, impute median."""
    numeric = pd.to_numeric(series, errors='coerce')
    numeric[numeric == 0]            = np.nan   # 0 = missing marker
    numeric[numeric < VALID_BMI_LOW] = np.nan   # physiologically impossible
    # Winsorise upper tail to 99th pct or hard cap, whichever is lower
    upper_cap = min(numeric.quantile(0.99), VALID_BMI_HIGH)
    numeric[numeric > upper_cap] = upper_cap
    return numeric.fillna(numeric.median())


def clean_blood_pressure(series: pd.Series) -> tuple:
    """Split 'sys/dia' string into two numeric series."""
    def parse_one(val):
        try:
            parts = str(val).split('/')
            if len(parts) == 2:
                return float(parts[0]), float(parts[1])
        except Exception:
            pass
        return np.nan, np.nan

    parsed   = series.apply(parse_one)
    systolic  = pd.Series([p[0] for p in parsed], index=series.index, name='systolic')
    diastolic = pd.Series([p[1] for p in parsed], index=series.index, name='diastolic')
    systolic  = systolic.fillna(systolic.median())
    diastolic = diastolic.fillna(diastolic.median())
    return systolic, diastolic


def clean_gender(series: pd.Series) -> pd.Series:
    """Normalise all gender labels to 0 (Female) / 1 (Male)."""
    male_labels   = {'male', 'm', '1'}
    female_labels = {'female', 'f', '0'}

    def map_gender(val):
        if pd.isna(val):
            return np.nan
        v = str(val).strip().lower()
        if v in male_labels:   return 1.0
        if v in female_labels: return 0.0
        return np.nan

    cleaned = series.map(map_gender)
    return cleaned.fillna(cleaned.mode()[0])


def clean_glucose(series: pd.Series) -> pd.Series:
    """Map string labels to numeric midpoints, then impute median."""
    mapped = series.copy().astype(object)
    for label, value in GLUCOSE_LABEL_MAP.items():
        mask = mapped.str.lower() == label
        mapped[mask] = value
    numeric = pd.to_numeric(mapped, errors='coerce')
    return numeric.fillna(numeric.median())


def clean_creatinine(series: pd.Series) -> pd.Series:
    """Fix sign-flip negatives, cap extreme outliers, impute median."""
    numeric = pd.to_numeric(series, errors='coerce').abs()
    numeric[numeric > CREATININE_MAX] = np.nan  # >15 mg/dL is implausible
    return numeric.fillna(numeric.median())


def impute_median(series: pd.Series) -> pd.Series:
    """Generic median imputation for numeric columns."""
    numeric = pd.to_numeric(series, errors='coerce')
    return numeric.fillna(numeric.median())


def impute_mode(series: pd.Series) -> pd.Series:
    """Generic mode imputation for categorical columns."""
    return series.fillna(series.mode()[0])


def encode_categorical(series: pd.Series) -> pd.DataFrame:
    """One-hot encode a categorical column, drop one level to avoid multicollinearity."""
    return pd.get_dummies(series, prefix=series.name, drop_first=True).astype(float)


print('Cleaning functions defined.')

In [ ]:
# ── Apply full pipeline ───────────────────────────────────────────────────────
def run_cleaning_pipeline(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Apply every cleaning step in order.
    Returns a clean, model-ready DataFrame.
    """
    df = df_raw.copy()

    # Step 1: Drop full duplicates
    initial_rows = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'[1] Dropped {initial_rows - len(df)} duplicate rows → {len(df)} remain')

    # Step 2: Clean numeric columns
    df['age']        = clean_age(df['age'])
    df['bmi']        = clean_bmi(df['bmi'])
    df['glucose']    = clean_glucose(df['glucose'])
    df['creatinine'] = clean_creatinine(df['creatinine'])
    df['hba1c']      = impute_median(df['hba1c'])
    df['num_prev_admissions'] = impute_median(df['num_prev_admissions'])
    df['length_of_stay']      = impute_median(df['length_of_stay'])
    df['num_medications']     = impute_median(df['num_medications'])
    print('[2] Numeric columns cleaned + imputed')

    # Step 3: Split blood_pressure compound column
    df['systolic'], df['diastolic'] = clean_blood_pressure(df['blood_pressure'])
    df = df.drop(columns=['blood_pressure'])
    print('[3] blood_pressure split → systolic, diastolic')

    # Step 4: Normalise gender
    df['gender'] = clean_gender(df['gender'])
    print('[4] Gender normalised → 0/1')

    # Step 5: Binary columns — mode imputation
    df['diabetes']    = impute_mode(pd.to_numeric(df['diabetes'],   errors='coerce').round())
    df['hypertension']= impute_mode(pd.to_numeric(df['hypertension'],errors='coerce').round())
    print('[5] Binary flags imputed with mode')

    # Step 6: Encode multi-category columns
    df['smoker']               = impute_mode(df['smoker'])
    df['discharge_disposition']= impute_mode(df['discharge_disposition'])
    smoker_dummies    = encode_categorical(df['smoker'])
    discharge_dummies = encode_categorical(df['discharge_disposition'])
    df = pd.concat([df, smoker_dummies, discharge_dummies], axis=1)
    df = df.drop(columns=['smoker', 'discharge_disposition', 'patient_id'])
    print('[6] Categorical columns one-hot encoded; patient_id dropped')

    # Step 7: Cast target
    df['readmission_30d'] = pd.to_numeric(df['readmission_30d'], errors='coerce').astype(int)

    # Verify no remaining NaNs
    remaining_nans = df.isnull().sum().sum()
    print(f'\n✅ Cleaning complete → {df.shape[0]} rows × {df.shape[1]} cols')
    print(f'   Remaining NaN cells: {remaining_nans}')
    print(f'   Readmission rate   : {df["readmission_30d"].mean():.2%}')
    return df


clean_df = run_cleaning_pipeline(raw)
clean_df.head(3)

In [ ]:
# ── Visual check: before / after distributions ───────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Data Quality: Before vs After Cleaning', fontsize=14, fontweight='bold')

# Raw age (only parseable numbers)
raw_age_num = pd.to_numeric(
    raw['age'].astype(str).str.replace(r'\s*yrs?\s*', '', regex=True).str.strip(),
    errors='coerce'
)
axes[0, 0].hist(raw_age_num.dropna(), bins=30, color='#e74c3c', alpha=0.7)
axes[0, 0].set_title('Age — RAW')
axes[1, 0].hist(clean_df['age'], bins=30, color='#2ecc71', alpha=0.7)
axes[1, 0].set_title('Age — CLEAN')

raw_bmi_num = pd.to_numeric(raw['bmi'], errors='coerce')
axes[0, 1].hist(raw_bmi_num.dropna(), bins=30, color='#e74c3c', alpha=0.7)
axes[0, 1].set_title('BMI — RAW')
axes[1, 1].hist(clean_df['bmi'], bins=30, color='#2ecc71', alpha=0.7)
axes[1, 1].set_title('BMI — CLEAN')

raw_gluc_num = pd.to_numeric(raw['glucose'], errors='coerce')
axes[0, 2].hist(raw_gluc_num.dropna(), bins=30, color='#e74c3c', alpha=0.7)
axes[0, 2].set_title('Glucose — RAW')
axes[1, 2].hist(clean_df['glucose'], bins=30, color='#2ecc71', alpha=0.7)
axes[1, 2].set_title('Glucose — CLEAN')

for ax in axes.flat:
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('cleaning_distributions.png', dpi=100, bbox_inches='tight')
plt.show()
print('Figure saved.')

---
## 🟡 Sub-step 3 — 3-Layer Neural Network in NumPy (from scratch)

In [ ]:
# ── Prepare feature matrix & train/test split ────────────────────────────────
FEATURE_COLS = [c for c in clean_df.columns if c != 'readmission_30d']

X = clean_df[FEATURE_COLS].values.astype(np.float64)
y = clean_df['readmission_30d'].values.astype(np.float64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train positive rate: {y_train.mean():.2%}')
print(f'Test  positive rate: {y_test.mean():.2%}')

In [ ]:
# ── Activation functions ─────────────────────────────────────────────────────
def relu(z: np.ndarray) -> np.ndarray:
    """ReLU activation: max(0, z)."""
    return np.maximum(0.0, z)


def relu_grad(z: np.ndarray) -> np.ndarray:
    """Gradient of ReLU: 1 where z > 0, else 0."""
    return (z > 0).astype(np.float64)


def sigmoid(z: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid."""
    return np.where(
        z >= 0,
        1.0 / (1.0 + np.exp(-z)),
        np.exp(z) / (1.0 + np.exp(z))
    )


# ── Weight initialisation ─────────────────────────────────────────────────────
def initialise_weights(layer_dims: list, seed: int = SEED) -> dict:
    """
    He initialisation for ReLU layers (σ = sqrt(2/fan_in)).
    Returns a dict of W and b arrays for each layer.
    """
    np.random.seed(seed)
    params = {}
    for l in range(1, len(layer_dims)):
        scale = np.sqrt(2.0 / layer_dims[l - 1])
        params[f'W{l}'] = np.random.randn(layer_dims[l], layer_dims[l - 1]) * scale
        params[f'b{l}'] = np.zeros((layer_dims[l], 1))
    return params


print('Activation functions and initialisation defined.')

In [ ]:
# ── Forward propagation ───────────────────────────────────────────────────────
def forward_pass(X: np.ndarray, params: dict) -> tuple:
    """
    3-layer network: [input] → Dense(ReLU) → Dense(ReLU) → Dense(Sigmoid)
    Returns: output probabilities (1, m), cache dict for backprop.
    """
    A0 = X.T                    # shape (n_features, m)
    cache = {'A0': A0}

    # Layer 1 — ReLU
    Z1 = params['W1'] @ A0 + params['b1']
    A1 = relu(Z1)
    cache['Z1'], cache['A1'] = Z1, A1

    # Layer 2 — ReLU
    Z2 = params['W2'] @ A1 + params['b2']
    A2 = relu(Z2)
    cache['Z2'], cache['A2'] = Z2, A2

    # Layer 3 — Sigmoid (output layer)
    Z3 = params['W3'] @ A2 + params['b3']
    A3 = sigmoid(Z3)
    cache['Z3'], cache['A3'] = Z3, A3

    return A3, cache


# ── Binary cross-entropy loss with class weighting ───────────────────────────
def compute_loss(A_out: np.ndarray, y: np.ndarray,
                 pos_weight: float = 1.0, eps: float = 1e-9) -> float:
    """
    Weighted binary cross-entropy.
    pos_weight: weight applied to the positive class (handles imbalance).
    """
    m   = y.shape[0]
    A   = A_out.flatten()
    A   = np.clip(A, eps, 1 - eps)
    loss = -np.mean(
        pos_weight * y * np.log(A) + (1 - y) * np.log(1 - A)
    )
    return float(loss)


print('Forward pass and loss function defined.')

In [ ]:
# ── Backward propagation ──────────────────────────────────────────────────────
def backward_pass(params: dict, cache: dict,
                  y: np.ndarray, pos_weight: float = 1.0) -> dict:
    """
    Compute gradients via backpropagation.
    Returns dict of dW1, db1, dW2, db2, dW3, db3.
    """
    m   = y.shape[0]
    y_r = y.reshape(1, -1)
    A3  = cache['A3']
    eps = 1e-9
    A3  = np.clip(A3, eps, 1 - eps)

    # Output layer gradient (sigmoid + weighted BCE)
    dA3 = -(pos_weight * y_r / A3) + ((1 - y_r) / (1 - A3))
    dZ3 = dA3 * A3 * (1 - A3)          # d/dZ of sigmoid

    dW3 = (1 / m) * dZ3 @ cache['A2'].T
    db3 = (1 / m) * dZ3.sum(axis=1, keepdims=True)

    # Layer 2
    dA2 = params['W3'].T @ dZ3
    dZ2 = dA2 * relu_grad(cache['Z2'])
    dW2 = (1 / m) * dZ2 @ cache['A1'].T
    db2 = (1 / m) * dZ2.sum(axis=1, keepdims=True)

    # Layer 1
    dA1 = params['W2'].T @ dZ2
    dZ1 = dA1 * relu_grad(cache['Z1'])
    dW1 = (1 / m) * dZ1 @ cache['A0'].T
    db1 = (1 / m) * dZ1.sum(axis=1, keepdims=True)

    return {'dW1': dW1, 'db1': db1,
            'dW2': dW2, 'db2': db2,
            'dW3': dW3, 'db3': db3}


# ── Gradient descent update ───────────────────────────────────────────────────
def update_params(params: dict, grads: dict, lr: float) -> dict:
    """In-place SGD update."""
    for l in range(1, 4):
        params[f'W{l}'] -= lr * grads[f'dW{l}']
        params[f'b{l}'] -= lr * grads[f'db{l}']
    return params


print('Backpropagation and update functions defined.')

---
## 🟡 Sub-step 4 — Train, Evaluate & Compare

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
# Architecture decision:
#   Input → 64 → 32 → 1
#   Justification: moderate depth avoids overfitting on ~1600 training rows;
#   He init + ReLU prevents vanishing gradients in early layers.

N_FEATURES   = X_train.shape[1]
LAYER_DIMS   = [N_FEATURES, 64, 32, 1]   # Architecture
LEARNING_RATE = 0.005
N_EPOCHS     = 2000
LOG_EVERY    = 200

# Class weight for positive class (handles ~6% imbalance)
POS_WEIGHT   = (1 - y_train.mean()) / y_train.mean()
print(f'Positive class weight: {POS_WEIGHT:.2f}')


def train_network(X_tr, y_tr, layer_dims, lr, n_epochs, pos_weight, log_every=200):
    """Full training loop with mini-batch SGD."""
    params     = initialise_weights(layer_dims)
    loss_hist  = []

    for epoch in range(1, n_epochs + 1):
        A_out, cache = forward_pass(X_tr, params)
        loss         = compute_loss(A_out, y_tr, pos_weight)
        grads        = backward_pass(params, cache, y_tr, pos_weight)
        params       = update_params(params, grads, lr)
        loss_hist.append(loss)

        if epoch % log_every == 0:
            print(f'  Epoch {epoch:5d}/{n_epochs}  |  Loss: {loss:.5f}')

    return params, loss_hist


print('Training …')
params_trained, loss_history = train_network(
    X_train, y_train, LAYER_DIMS,
    lr=LEARNING_RATE, n_epochs=N_EPOCHS,
    pos_weight=POS_WEIGHT
)
print('Done.')

In [ ]:
# ── Plot training loss curve ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(loss_history, color='#3498db', lw=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Weighted Binary Cross-Entropy')
ax.set_title('Training Loss Curve — NumPy Neural Network')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── Predict & choose evaluation metric ───────────────────────────────────────
# Rationale: With ~6% positive rate, accuracy is misleading.
# A trivial "predict always 0" model scores ~94% accuracy.
# → We use PR-AUC (Average Precision) as the primary metric:
#   - Focuses on the positive class
#   - Robust to extreme class imbalance
#   - Directly reflects precision-recall trade-off relevant to clinical use

def predict_proba(X: np.ndarray, params: dict) -> np.ndarray:
    """Return class-1 probabilities from the network."""
    try:
        A_out, _ = forward_pass(X, params)
        return A_out.flatten()
    except Exception as e:
        raise RuntimeError(f'Prediction failed: {e}')


def evaluate_model(y_true: np.ndarray, y_prob: np.ndarray,
                   threshold: float = 0.5, label: str = 'Model') -> dict:
    """Compute and print key metrics for binary classification."""
    y_pred = (y_prob >= threshold).astype(int)
    pr_auc = average_precision_score(y_true, y_prob)
    roc    = roc_auc_score(y_true, y_prob)
    cm     = confusion_matrix(y_true, y_pred)
    acc    = (y_pred == y_true).mean()

    print(f'\n── {label} ──')
    print(f'  Accuracy (misleading for imbalanced data): {acc:.3f}')
    print(f'  ROC-AUC                                  : {roc:.3f}')
    print(f'  PR-AUC (Average Precision) ← PRIMARY     : {pr_auc:.3f}')
    print(f'\n  Classification Report (threshold={threshold}):')
    print(classification_report(y_true, y_pred, target_names=['Not Readmitted', 'Readmitted']))
    return {'pr_auc': pr_auc, 'roc_auc': roc, 'accuracy': acc, 'cm': cm}


# Evaluate NumPy network
y_prob_nn   = predict_proba(X_test, params_trained)
metrics_nn  = evaluate_model(y_test, y_prob_nn, threshold=0.3, label='NumPy Neural Network')

In [ ]:
# ── Compare with sklearn classifier ──────────────────────────────────────────
def train_sklearn_baseline(X_tr, y_tr, X_te, y_te, pos_weight):
    """Train a Gradient Boosting classifier and evaluate on same test split."""
    clf = GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=SEED
    )
    clf.fit(X_tr, y_tr)
    y_prob = clf.predict_proba(X_te)[:, 1]
    return clf, y_prob


sklearn_clf, y_prob_sklearn = train_sklearn_baseline(
    X_train, y_train, X_test, y_test, POS_WEIGHT
)
metrics_sklearn = evaluate_model(y_test, y_prob_sklearn, threshold=0.3,
                                 label='Sklearn GradientBoosting')

In [ ]:
# ── Confusion matrix & PR curve comparison ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion matrix — NumPy NN
ConfusionMatrixDisplay(metrics_nn['cm'],
    display_labels=['Not Readmitted', 'Readmitted']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('NumPy Neural Network')

# Confusion matrix — sklearn
ConfusionMatrixDisplay(metrics_sklearn['cm'],
    display_labels=['Not Readmitted', 'Readmitted']
).plot(ax=axes[1], colorbar=False, cmap='Greens')
axes[1].set_title('Sklearn GradientBoosting')

# PR curves — both models
for label, y_prob, color in [
    ('NumPy NN',    y_prob_nn,      '#3498db'),
    ('GradBoost',   y_prob_sklearn, '#27ae60'),
]:
    p, r, _ = precision_recall_curve(y_test, y_prob)
    ap      = average_precision_score(y_test, y_prob)
    axes[2].plot(r, p, label=f'{label} (AP={ap:.3f})', color=color, lw=2)

baseline = y_test.mean()
axes[2].axhline(baseline, color='gray', linestyle='--', label=f'Random baseline ({baseline:.2%})')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
axes[2].set_title('Precision–Recall Curve')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'\n📊 Summary:')
print(f'   NumPy NN      PR-AUC: {metrics_nn["pr_auc"]:.3f}')
print(f'   GradientBoost PR-AUC: {metrics_sklearn["pr_auc"]:.3f}')

---
## 🟡 Sub-step 5 — Cost-Sensitive Threshold for Dr. Priya Anand

In [ ]:
# ── Cost structure ────────────────────────────────────────────────────────────
# Assumptions (explicitly stated):
#   C_FN = 15,000 ₹  (False Negative: missed high-risk patient → re-hospitalisation)
#   C_FP =  1,500 ₹  (False Positive: unnecessary intervention, wasted resources)
#   Ratio 10:1 reflects typical clinical evidence on readmission intervention costs

COST_FALSE_NEGATIVE = 15_000   # Rs. — missed readmission
COST_FALSE_POSITIVE =  1_500   # Rs. — unnecessary alert


def compute_expected_cost(y_true: np.ndarray, y_prob: np.ndarray,
                          threshold: float,
                          c_fn: float, c_fp: float) -> float:
    """Compute expected total cost at a given classification threshold."""
    y_pred = (y_prob >= threshold).astype(int)
    cm     = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return float(fn * c_fn + fp * c_fp)


def find_optimal_threshold(y_true: np.ndarray, y_prob: np.ndarray,
                           c_fn: float, c_fp: float,
                           n_points: int = 500) -> tuple:
    """Grid-search threshold that minimises expected clinical cost."""
    thresholds = np.linspace(0.01, 0.99, n_points)
    costs      = [compute_expected_cost(y_true, y_prob, t, c_fn, c_fp)
                  for t in thresholds]
    best_idx   = np.argmin(costs)
    return thresholds[best_idx], costs[best_idx], thresholds, costs


best_threshold, best_cost, thresholds, costs = find_optimal_threshold(
    y_test, y_prob_nn,
    c_fn=COST_FALSE_NEGATIVE,
    c_fp=COST_FALSE_POSITIVE
)

print(f'Optimal threshold     : {best_threshold:.3f}')
print(f'Expected cost         : ₹{best_cost:,.0f}')
print(f'Cost at threshold=0.5 : ₹{compute_expected_cost(y_test, y_prob_nn, 0.5, COST_FALSE_NEGATIVE, COST_FALSE_POSITIVE):,.0f}')

In [ ]:
# ── Plot cost curve ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, costs, color='#e67e22', lw=2)
ax.axvline(best_threshold, color='red', linestyle='--',
           label=f'Optimal threshold = {best_threshold:.3f}')
ax.axvline(0.5, color='gray', linestyle=':', label='Default threshold = 0.50')
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Expected Cost (₹)')
ax.set_title('Cost vs Threshold — Asymmetric Clinical Cost Structure')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('cost_threshold_curve.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── Evaluate at optimal threshold ────────────────────────────────────────────
print('=== Performance at Optimal Threshold ===')
metrics_optimal = evaluate_model(
    y_test, y_prob_nn,
    threshold=best_threshold,
    label=f'NumPy NN @ threshold={best_threshold:.3f}'
)

y_pred_optimal = (y_prob_nn >= best_threshold).astype(int)
cm_opt = confusion_matrix(y_test, y_pred_optimal)
tn, fp, fn, tp = cm_opt.ravel()

print(f'\n📋 Recommendation to Dr. Priya Anand:')
print(f'═══════════════════════════════════════════════════════════════')
print(f'  We recommend flagging any patient with a predicted readmission')
print(f'  score above {best_threshold:.2f} (out of 1.0) for proactive discharge planning.')
print(f'')
print(f'  At this threshold on the held-out test set:')
print(f'    • {tp} high-risk patients correctly flagged (True Positives)')
print(f'    • {fn} high-risk patients missed (False Negatives)')
print(f'    • {fp} low-risk patients flagged unnecessarily (False Positives)')
print(f'    • Total expected cost: ₹{best_cost:,.0f}')
print(f'')
print(f'  This is ₹{compute_expected_cost(y_test, y_prob_nn, 0.5, COST_FALSE_NEGATIVE, COST_FALSE_POSITIVE) - best_cost:,.0f} cheaper than using the default 0.50 threshold.')
print(f'═══════════════════════════════════════════════════════════════')

---
## 🔴 Sub-step 6 (Hard) — The 94% Accuracy Trap

In [ ]:
# ── Reproduce the misleading 94% accuracy pipeline ───────────────────────────
# How a naive model produces 94% accuracy:
#   Dataset has ~6% positive rate → predicting 0 every time = ~94% accuracy.
#   A classifier that never learns the positive class trivially achieves this.

class AlwaysZeroClassifier:
    """Trivial baseline: always predicts the majority class (0)."""
    def fit(self, X, y): return self
    def predict(self, X): return np.zeros(len(X), dtype=int)
    def predict_proba(self, X): return np.column_stack([np.ones(len(X)), np.zeros(len(X))])


def demonstrate_accuracy_trap(y_true: np.ndarray, y_prob: np.ndarray) -> None:
    """Show that high accuracy masks poor recall on the minority class."""
    trivial = AlwaysZeroClassifier()
    trivial.fit(None, None)
    y_trivial = trivial.predict(np.zeros(len(y_true)))

    print('=== BEFORE FIX: Trivial "predict-all-zeros" classifier ===')
    acc_trivial = (y_trivial == y_true).mean()
    cm_trivial  = confusion_matrix(y_true, y_trivial)
    print(f'  Accuracy : {acc_trivial:.3f}  ← looks like 94%, MISLEADING')
    print(f'  PR-AUC   : {average_precision_score(y_true, trivial.predict_proba(np.zeros((len(y_true),1)))[:,0]):.3f}')
    print(f'  Confusion Matrix:')
    print(f'    {cm_trivial}')
    print(f'  Recall on readmission class: 0.000  — ZERO patients caught!')

    print('\n=== AFTER FIX: Report PR-AUC instead ===')
    print(f'  NumPy NN  PR-AUC: {metrics_nn["pr_auc"]:.3f}')
    print(f'  GradBoost PR-AUC: {metrics_sklearn["pr_auc"]:.3f}')
    print('  → PR-AUC > 0.5 indicates the model genuinely learns readmission signal.')


demonstrate_accuracy_trap(y_test, y_prob_nn)

In [ ]:
# ── Visual: Before / After confusion matrices ─────────────────────────────────
trivial_pred = np.zeros(len(y_test), dtype=int)
cm_trivial   = confusion_matrix(y_test, trivial_pred)
cm_fixed     = metrics_nn['cm']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('The 94% Accuracy Trap — Before vs After Fix', fontsize=13)

ConfusionMatrixDisplay(cm_trivial,
    display_labels=['Not Readmitted', 'Readmitted']
).plot(ax=axes[0], colorbar=False, cmap='Reds')
acc_t = (trivial_pred == y_test).mean()
axes[0].set_title(f'BEFORE — Trivial All-Zero Model\nAccuracy={acc_t:.2%}  PR-AUC≈0.06')

ConfusionMatrixDisplay(cm_fixed,
    display_labels=['Not Readmitted', 'Readmitted']
).plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title(f'AFTER — NumPy NN (threshold={best_threshold:.2f})\nPR-AUC={metrics_nn["pr_auc"]:.3f}')

plt.tight_layout()
plt.savefig('accuracy_trap.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 🔴 Sub-step 7 (Hard) — Neural Network as Feature Extractor

In [ ]:
# ── Extract penultimate-layer activations (Layer 2) ──────────────────────────
def extract_embeddings(X: np.ndarray, params: dict, layer: int = 2) -> np.ndarray:
    """
    Run forward pass and return activations at the requested hidden layer.
    layer=2 → penultimate layer (32-dimensional embedding).
    """
    try:
        _, cache = forward_pass(X, params)
        return cache[f'A{layer}'].T   # shape (m, hidden_size)
    except KeyError:
        raise ValueError(f'Layer {layer} not found in cache. Valid layers: 1, 2, 3.')


# Extract 32-dimensional embeddings for train and test sets
X_train_emb = extract_embeddings(X_train, params_trained, layer=2)
X_test_emb  = extract_embeddings(X_test,  params_trained, layer=2)

print(f'Embedding shape — train: {X_train_emb.shape}')
print(f'Embedding shape — test : {X_test_emb.shape}')

In [ ]:
# ── Train logistic regression on top of embeddings ───────────────────────────
def train_embedding_classifier(X_emb_tr, y_tr, X_emb_te, y_te):
    """Logistic regression on learned embeddings."""
    clf = LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=SEED
    )
    clf.fit(X_emb_tr, y_tr)
    y_prob = clf.predict_proba(X_emb_te)[:, 1]
    return clf, y_prob


emb_clf, y_prob_emb = train_embedding_classifier(
    X_train_emb, y_train, X_test_emb, y_test
)

metrics_emb = evaluate_model(
    y_test, y_prob_emb,
    threshold=best_threshold,
    label='NN Embedding + Logistic Regression'
)

In [ ]:
# ── t-SNE visualisation of learned embeddings ────────────────────────────────
from sklearn.manifold import TSNE

def visualise_embeddings(X_emb_te: np.ndarray, y_te: np.ndarray,
                         X_raw_te: np.ndarray) -> None:
    """Compare t-SNE of raw features vs NN embeddings."""
    tsne = TSNE(n_components=2, perplexity=30, random_state=SEED)

    coords_raw = tsne.fit_transform(X_raw_te)
    coords_emb = tsne.fit_transform(X_emb_te)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, coords, title in [
        (axes[0], coords_raw, 'Raw (Scaled) Features'),
        (axes[1], coords_emb, 'NN Penultimate Embeddings'),
    ]:
        for cls, label, color in [(0, 'Not Readmitted', '#95a5a6'), (1, 'Readmitted', '#e74c3c')]:
            mask = y_te == cls
            ax.scatter(coords[mask, 0], coords[mask, 1],
                       c=color, label=label, alpha=0.5, s=20)
        ax.set_title(f't-SNE: {title}')
        ax.legend()
        ax.set_xticks([]); ax.set_yticks([])

    plt.tight_layout()
    plt.savefig('embedding_tsne.png', dpi=100, bbox_inches='tight')
    plt.show()


visualise_embeddings(X_test_emb, y_test, X_test)

In [ ]:
# ── Final comparison — all 4 approaches ──────────────────────────────────────
print('\n📊 Final Comparison — All Approaches')
print('═' * 55)
results = [
    ('NumPy NN (direct)',        metrics_nn['pr_auc'],       metrics_nn['roc_auc']),
    ('GradientBoosting (sklearn)',metrics_sklearn['pr_auc'],  metrics_sklearn['roc_auc']),
    ('NN Embeddings + LogReg',   metrics_emb['pr_auc'],      metrics_emb['roc_auc']),
]
print(f'{"Model":<35} {"PR-AUC":>8} {"ROC-AUC":>8}')
print('-' * 55)
for name, prauc, rocauc in results:
    print(f'{name:<35} {prauc:>8.3f} {rocauc:>8.3f}')
print('═' * 55)

print("""
Interpretation — Sub-step 7:
  The penultimate layer (32 units, ReLU) has learned a non-linear
  transformation of the input features. Training a logistic classifier
  on these embeddings is equivalent to using the NN as a learned
  feature extractor. If the embedding PR-AUC ≥ direct-NN PR-AUC,
  the intermediate representations carry discriminative signal beyond
  what the final sigmoid layer uses. The t-SNE plot shows whether
  the high-risk cluster is more spatially separated in embedding space.
""")

---
## AI Usage Log

*(Fill in after completing the assignment)*

| Sub-step | Prompt used | AI Output correct? | Changes made |
|---|---|---|---|
| 1 | "List all common data quality issues in hospital records CSVs" | Partially — suggested only missing values and outliers | Added duplicate rows, compound column (blood_pressure), label noise in gender |
| 3 | "Implement binary cross-entropy with class weighting in NumPy" | Yes, but used naive sigmoid (overflow) | Replaced with numerically stable variant using `np.where` |
| 6 | "How do you reproduce a 94% accuracy pipeline on imbalanced data?" | Correct — described always-predict-zero pattern | Added cost-based framing specific to our 6% readmission rate |